In [ ]:
import gymnasium as gym
import numpy as np
import pygame
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
import io

# --------------------------
# Parameters
# --------------------------
bins = 20
episodes = 500
alpha = 0.1
gamma = 0.99
epsilon = 0.1

# --------------------------
# Discretization
# --------------------------
state_space = [
    np.linspace(-4.8, 4.8, bins),
    np.linspace(-5, 5, bins),
    np.linspace(-0.418, 0.418, bins),
    np.linspace(-5, 5, bins)
]

def discretize(obs):
    return tuple(np.digitize(o, b) for o, b in zip(obs, state_space))

# --------------------------
# Q-table
# --------------------------
q_table = np.zeros([bins]*4 + [2])  # CartPole action space is 2

# --------------------------
# Gym environment
# --------------------------
env = gym.make("CartPole-v1", render_mode="rgb_array")  # get RGB frames for pygame

# --------------------------
# Initialize pygame window
# --------------------------
pygame.init()
env_width, env_height = 600, 400
window = pygame.display.set_mode((env_width*2, env_height))
pygame.display.set_caption("CartPole + Live Reward Plot")

# --------------------------
# Reward tracking
# --------------------------
reward_history = []

def moving_average(x, window=20):
    if len(x) < window:
        return x
    return np.convolve(x, np.ones(window)/window, mode='valid')

def plot_rewards(rewards):
    """Return a Pygame surface with the reward plot"""
    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(rewards, alpha=0.3, label="Episode Reward")
    if len(rewards) >= 20:
        ma = moving_average(rewards, window=20)
        ax.plot(range(19, len(rewards)), ma, color='red', label="Moving Avg")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Total Reward")
    ax.set_title("Reward Progress")
    ax.legend()
    fig.tight_layout()

    canvas = FigureCanvasAgg(fig)
    canvas.draw()

    # Use buffer_rgba and convert to RGB
    buf = np.asarray(canvas.buffer_rgba())
    buf = buf[:, :, :3]  # drop alpha channel
    plt.close(fig)

    # Convert to Pygame surface
    return pygame.surfarray.make_surface(np.flipud(np.rot90(buf)))



# --------------------------
# Training loop with live visualization
# --------------------------
for ep in range(episodes):
    obs, _ = env.reset()
    state = discretize(obs)
    done = False
    total_reward = 0

    while not done:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                done = True
                break

        # Epsilon-greedy action
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state])

        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state = discretize(next_obs)

        # Q-learning update
        q_table[state][action] += alpha * (reward + gamma * np.max(q_table[next_state]) - q_table[state][action])
        state = next_state
        total_reward += reward

        # --------------------------
        # Render environment
        # --------------------------
        frame = env.render()
        surf = pygame.surfarray.make_surface(np.flipud(np.rot90(frame)))
        window.blit(pygame.transform.scale(surf, (env_width, env_height)), (0,0))

        # Render reward plot
        reward_surf = pygame.transform.scale(plot_rewards(reward_history + [total_reward]), (env_width, env_height))
        window.blit(reward_surf, (env_width, 0))

        pygame.display.update()

    reward_history.append(total_reward)

    if ep % 50 == 0:
        print(f"Episode {ep}, Reward: {total_reward}")

# Save Q-table
np.save("q_table.npy", q_table)
print("Training complete!")

pygame.quit()
env.close()


Episode 0, Reward: 9.0


In [3]:
import pygame
import math
import sys

# Initialize Pygame
pygame.init()

# Screen dimensions
WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Inverted Pendulum Game")

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED = (255, 0, 0)

# Clock
clock = pygame.time.Clock()
FPS = 60

# Pendulum and cart parameters
cart_width = 100
cart_height = 20
pole_length = 200
cart_x = WIDTH // 2
cart_y = HEIGHT - 100

# Physics parameters
theta = math.pi / 6  # initial angle (30 degrees)
theta_dot = 0  # angular velocity
cart_v = 0  # cart velocity
gravity = 9.8
force = 0  # applied force
mass_cart = 1.0
mass_pole = 0.1
dt = 1 / FPS

# Control parameters
cart_speed = 500  # pixels per second

def draw_cart_pole(cart_x, theta):
    # Draw cart
    pygame.draw.rect(screen, BLACK, (cart_x - cart_width // 2, cart_y, cart_width, cart_height))
    
    # Calculate pole tip position
    pole_x = cart_x + pole_length * math.sin(theta)
    pole_y = cart_y - pole_length * math.cos(theta)
    
    # Draw pole
    pygame.draw.line(screen, RED, (cart_x, cart_y), (pole_x, pole_y), 5)
    
def update_physics(force):
    global theta, theta_dot, cart_x, cart_v
    
    # Simplified physics equations for inverted pendulum on a cart
    theta_acc = (gravity * math.sin(theta) + math.cos(theta) * (-force - mass_pole * pole_length * theta_dot**2 * math.sin(theta)) / (mass_cart + mass_pole)) / \
                (pole_length * (4/3 - mass_pole * math.cos(theta)**2 / (mass_cart + mass_pole)))
    
    theta_dot += theta_acc * dt
    theta += theta_dot * dt
    
    cart_acc = (force + mass_pole * pole_length * (theta_dot**2 * math.sin(theta) - theta_acc * math.cos(theta))) / (mass_cart + mass_pole)
    cart_v += cart_acc * dt
    cart_x += cart_v * dt
    
    # Keep cart on screen
    if cart_x < cart_width // 2:
        cart_x = cart_width // 2
        cart_v = 0
    elif cart_x > WIDTH - cart_width // 2:
        cart_x = WIDTH - cart_width // 2
        cart_v = 0

# Game loop
running = True
while running:
    screen.fill(WHITE)
    
    # Handle events
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
    
    keys = pygame.key.get_pressed()
    force = 0
    if keys[pygame.K_LEFT]:
        force = -cart_speed
    elif keys[pygame.K_RIGHT]:
        force = cart_speed
    
    # Update physics
    update_physics(force)
    
    # Draw everything
    draw_cart_pole(cart_x, theta)
    
    # Check if pole fell
    if abs(theta) > math.pi/2:
        font = pygame.font.SysFont(None, 60)
        text = font.render("Game Over!", True, RED)
        screen.blit(text, (WIDTH//2 - 150, HEIGHT//2))
    
    pygame.display.flip()
    clock.tick(FPS)

pygame.quit()
sys.exit()


SystemExit: 